##### Create a round robin matches for group of 3 teams

In [0]:
%sql
With teams as 
(
    Select 'India' as team_name
    union all
    Select 'Australia' 
    union all
    Select 'England'
),
pairs as 
(
    select t1.team_name as team1, t2.team_name as team2
    from teams t1
    join teams t2
      on t1.team_name < t2.team_name
),
rounds as
(
    select 1 as cycle_number
    union all  
    select 2
    union all
    select 3
)

select cycle_number, team1,team2,
row_number() over (order by cycle_number,team1,team2 ) as round_number
from pairs
cross join rounds
on 1=1
order by cycle_number

cycle_number,team1,team2,round_number
1,Australia,England,1
1,Australia,India,2
1,England,India,3
2,Australia,England,4
2,Australia,India,5
2,England,India,6
3,Australia,England,7
3,Australia,India,8
3,England,India,9


#### Employee Shift Rotation

You have 10 employees and 3 shifts (Morning, Afternoon, Night). Write a query to assign employees to shifts in a rotating fashion across 7 days.  

In [0]:
%sql
WITH employees AS (
    SELECT 'Alice' AS employee_name UNION ALL 
    SELECT 'Bob' UNION ALL
    SELECT 'Charlie' UNION ALL
    SELECT 'David' UNION ALL
    SELECT 'Eve' UNION ALL
    SELECT 'Frank' UNION ALL
    SELECT 'Grace' UNION ALL
    SELECT 'Heidi' UNION ALL
    SELECT 'Ivan' UNION ALL
    SELECT 'Judy'
),
employees_numbered AS (
    SELECT employee_name,
           ROW_NUMBER() OVER (ORDER BY employee_name) AS emp_no
    FROM employees
),
shifts AS (
    SELECT 0 AS shift_id, 'Morning' AS shift_name UNION ALL
    SELECT 1, 'Afternoon' UNION ALL
    SELECT 2, 'Evening'
),
working_days AS (
    SELECT 1 AS working_day_no, 'Monday' AS working_day UNION ALL
    SELECT 2, 'Tuesday' UNION ALL
    SELECT 3, 'Wednesday' UNION ALL
    SELECT 4, 'Thursday' UNION ALL
    SELECT 5, 'Friday' UNION ALL
    SELECT 6, 'Saturday' UNION ALL
    SELECT 7, 'Sunday'
)
SELECT 
    e.employee_name, 
    s.shift_name, 
    d.working_day
FROM employees_numbered e
CROSS JOIN working_days d
JOIN shifts s
  ON ((e.emp_no + d.working_day_no - 1) % 3) = s.shift_id
ORDER BY d.working_day_no, e.employee_name;


employee_name,shift_name,working_day
Alice,Afternoon,Monday
Bob,Evening,Monday
Charlie,Morning,Monday
David,Afternoon,Monday
Eve,Evening,Monday
Frank,Morning,Monday
Grace,Afternoon,Monday
Heidi,Evening,Monday
Ivan,Morning,Monday
Judy,Afternoon,Monday


In [0]:
# The list of column names
columns = ['emp_id', 'employee_name', 'department', 'salary', 'joining_date']

# The sample data structured as a list of rows (lists)
data = [
    [101, 'Alice Smith', 'HR', 60000, '2023-01-15'],
    [102, 'Bob Jones', 'Engineering', 95000, '2022-05-12'],
    [103, 'Charlie Brown', 'Finance', 75000, '2024-03-01'],
    [104, 'Diana Prince', 'Engineering', 110000, '2021-08-19'],
    [105, 'Evan Wright', 'Marketing', 68000, '2023-11-10'],
    [106, 'Fiona Gallagher', 'HR', 62000, '2025-02-01']
]

from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import SparkSession

df=spark.createDataFrame(data,columns)

df.createOrReplaceTempView("employee")

##### Given an array of strings, group anagrams together.

An anagram is a word formed by rearranging the letters of another word. For example, "eat", "tea", and "ate" are all anagrams of each other because they all use exactly one 'a', one 'e', and one 't'.

In [0]:
from collections import defaultdict

def groupAnagrams(strs):
    # Create a dictionary where the default value is an empty list
    anagram_map = defaultdict(list)
    
    for word in strs:
        # Sort the letters of the word to create a unique key
        # e.g., 'tea' -> ['e', 'a', 't'] -> sorted -> ['a', 'e', 't'] -> 'aet'
        sorted_key = "".join(sorted(word))
        
        # Append the original word to the list matching that key
        anagram_map[sorted_key].append(word)
        
    # Return just the grouped lists
    return list(anagram_map.values())

# Test it out
words = ["eat", "tea", "tan", "ate", "nat", "bat"]
print(groupAnagrams(words))


[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


##### Write a Python function to find the maximum repeating character in a string and its count, without using built-in libraries like Collections.

In [0]:
def max_repeating_char(s: str):
    if not s:
        return "", 0

    max_char = s[0]
    max_count = 1
    
    current_char = s[0]
    current_count = 1

    for i in range(1, len(s)):
        if s[i] == current_char:
            current_count += 1
        else:
            if current_count > max_count:
                max_count = current_count
                max_char = current_char
            current_char = s[i]
            current_count = 1
            
    # Final check for the last sequence in the string
    if current_count > max_count:
        max_count = current_count
        max_char = current_char

    return max_char, max_count

# Example Test
print(max_repeating_char("aaabbdddddcccccddddddd")) 

('d', 7)


##### Processing a Massive File via Generator (yield)
The Goal: Stream a file line by line without pulling the whole file into system memory.

In [0]:
def read_large_file(file_path):
    with open(file_path, 'r') as file:
        for line in file:
            yield line.strip()

# Usage in a processing pipeline
for clean_line in read_large_file('/Volumes/workspace/default/landing_volume/massive_50gb_data.txt'):
    # Process your line here without crashing the 8GB RAM machine
    if "ERROR" in clean_line:
        print(clean_line)

ERROR  : go check your file
ERROR: Name missing in the data


#####  3rd Highest Salary in Each Department

The Goal: Handle duplicate high salaries correctly using DENSE_RANK().

In [0]:
columns = ["employee_id", "employee_name", "department", "salary", "manager_id"]

data = [
    (101, "John Smith",      "IT",      135000, 201),
    (102, "Alice Johnson",   "IT",      85000, 201),
    (103, "Bob Williams",    "HR",      70000, 202),
    (104, "Carol Brown",     "HR",      75000, 202),
    (105, "David Miller",    "Finance", 90000, 203),
    (106, "Emma Davis",      "Finance", 80000, 203),
    (107, "Frank Wilson",    "IT",      85000, 201),
    (108, "Grace Taylor",    "Sales",   120000, 204),
    (109, "Henry Moore",     "Sales",   68000, 204),
    (110, "Ivy Anderson",    "Marketing",76000, 205),
    (201, "Michael Scott",   "IT",     130000, 301),
    (202, "Sarah Connor",    "HR",     120000, 301),
    (203, "Robert King",     "Finance",145000, 301),
    (204, "Nancy White",     "Sales",  118000, 301),
    (205, "James Clark",     "Marketing",122000, 301),
    (301, "Dilip M G",     "CEO",140000, None),
     (302, "Emma Davis Jr",      "Finance", 80000, 203),
     (302, "Emma Davis Jr",      "Finance", 80000, 203)
]

df= spark.createDataFrame(data,columns)
df.createOrReplaceTempView('employees')

In [0]:
%sql
WITH RankedSalaries AS (
    SELECT 
        employee_id,
        employee_name,
        department,
        salary,
        DENSE_RANK() OVER (PARTITION BY department ORDER BY salary DESC) as rnk
    FROM employees
)
SELECT employee_id,employee_name, department, salary
FROM RankedSalaries
WHERE rnk = 3;

employee_id,employee_name,department,salary
106,Emma Davis,Finance,80000
302,Emma Davis Jr,Finance,80000
302,Emma Davis Jr,Finance,80000
103,Bob Williams,HR,70000
102,Alice Johnson,IT,85000
107,Frank Wilson,IT,85000
109,Henry Moore,Sales,68000


##### Delete Exact Duplicate Records (No Primary Key)
The Goal: Keep only one instance of completely identical rows using a CTE.

In [0]:
%sql
-- Create a working copy of the employees table so the original data stays intact
-- (used later for dedup practice, so any changes land on employee_test, not employees)
create or replace table employee_test as
select * from employees;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Deduplicate employee_test: there is no primary key, so we identify "exact" duplicates
-- by partitioning on ALL columns — only rows identical in every column land in the same group.
-- ROW_NUMBER() assigns 1..n within each duplicate group, and keeping rn = 1 retains a single copy.
-- The result is written to a new table so the source (employee_test) remains untouched.
CREATE OR REPLACE TABLE employees_dedup AS
SELECT employee_id, employee_name, department, salary, manager_id  -- drop the helper column 'rn'
FROM
(
    SELECT *,
           -- Partition by every column: rows that match on all values are true duplicates.
           -- ORDER BY employee_id just makes the numbering deterministic within each group.
           ROW_NUMBER() OVER
           (
               PARTITION BY employee_id, employee_name, department, salary, manager_id
               ORDER BY employee_id
           ) rn
    FROM employee_test
)
WHERE rn = 1;  -- keep exactly one row per duplicate group

-- Verify the deduplicated result
select * from employees_dedup;

employee_id,employee_name,department,salary,manager_id
101,John Smith,IT,135000,201
102,Alice Johnson,IT,85000,201
103,Bob Williams,HR,70000,202
104,Carol Brown,HR,75000,202
105,David Miller,Finance,90000,203
106,Emma Davis,Finance,80000,203
107,Frank Wilson,IT,85000,201
108,Grace Taylor,Sales,120000,204
109,Henry Moore,Sales,68000,204
110,Ivy Anderson,Marketing,76000,205


In [0]:
%sql
-- After deduplication, the cleaned data currently lives in employees_dedup.
-- Here we move it back: first copy the deduplicated rows into employee_test
-- (replacing whatever employee_test held before),
create or replace table employee_test as select * from employees_dedup; 

-- then drop the temporary employees_dedup table since its contents are now preserved
-- in employee_test (Databricks SQL does not support RENAME, so copy-then-drop is the pattern).
drop table employees_dedup;

-- Verify the final state: employee_test now holds the deduplicated employee data
select * from employee_test; 

employee_id,employee_name,department,salary,manager_id
101,John Smith,IT,135000,201
102,Alice Johnson,IT,85000,201
103,Bob Williams,HR,70000,202
104,Carol Brown,HR,75000,202
105,David Miller,Finance,90000,203
106,Emma Davis,Finance,80000,203
107,Frank Wilson,IT,85000,201
108,Grace Taylor,Sales,120000,204
109,Henry Moore,Sales,68000,204
110,Ivy Anderson,Marketing,76000,205


###### Using simple DISTINCT

In [0]:
%sql
-- Deduplicate with simple DISTINCT: unlike the ROW_NUMBER() approach,
-- DISTINCT automatically collapses ALL columns at once, so any row that is an
-- exact duplicate of another (identical in every column) is reduced to a single copy.
-- No helper column is needed, but it can't keep or control WHICH copy survives --
-- it also cannot rank or selectively retain rows (that requires the window-function CTE).
CREATE OR REPLACE TABLE employees_clean AS
SELECT DISTINCT 
  employee_id,
  employee_name,
  department,
  salary,
  manager_id 
FROM employees;  -- source still contains the duplicate 'Emma Davis Jr' rows

-- Verify the deduplicated result: the exact duplicates should be gone
select * from employees_clean
order by employee_id;

employee_id,employee_name,department,salary,manager_id
101,John Smith,IT,135000,201
102,Alice Johnson,IT,85000,201
103,Bob Williams,HR,70000,202
104,Carol Brown,HR,75000,202
105,David Miller,Finance,90000,203
106,Emma Davis,Finance,80000,203
107,Frank Wilson,IT,85000,201
108,Grace Taylor,Sales,120000,204
109,Henry Moore,Sales,68000,204
110,Ivy Anderson,Marketing,76000,205


##### Given a list of integers, write a script to find the longest consecutive sequence of numbers.

In [0]:
def longest_consecutive_sequence(nums):
    # No numbers means no sequence to find
    if not nums:
        return []

    # Sort so that consecutive numbers sit next to each other
    # (duplicates also become adjacent, e.g. [1, 2, 2, 3])
    nums.sort()

    # Best (longest) run found so far — initially the first number by itself
    best_start = nums[0]
    best_length = 1

    # Current run being tracked while scanning the sorted list
    curr_start = nums[0]
    curr_length = 1

    for i in range(1, len(nums)):

        if nums[i] == nums[i-1]:
            # Duplicate: skip it without breaking the current run
            continue

        elif nums[i] == nums[i-1] + 1:
            # Directly consecutive: extend the current run
            curr_length += 1

        else:
            # Gap found: the current run has ended.
            # Update the best run if the current one is longer
            if curr_length > best_length:
                best_length = curr_length
                best_start = curr_start

            # Always start a fresh run from the current number
            curr_start = nums[i]
            curr_length = 1

    # After the loop ends, check the final run too
    # (it never hit a gap, so it was never compared inside the loop)
    if curr_length > best_length:
        best_length = curr_length
        best_start = curr_start

    # Rebuild the winning sequence from its start and length
    return list(range(best_start, best_start + best_length))


# Test: the longest consecutive run here is 100-105
numbers = [100,200,3,4,2,1,5,101,102, 103,104,105]
print(longest_consecutive_sequence(numbers))

[100, 101, 102, 103, 104, 105]


## Pyspark

##### Remove duplicate

In [0]:
df.tail(2)

[Row(employee_id=302, employee_name='Emma Davis Jr', department='Finance', salary=80000, manager_id=203),
 Row(employee_id=302, employee_name='Emma Davis Jr', department='Finance', salary=80000, manager_id=203)]

In [0]:
# Remove exact duplicate rows using a window function (PySpark equivalent of the SQL CTE approach above)
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Partition by ALL columns: only rows identical in every column land in the same group,
# so they are true duplicates (no primary key exists to tell them apart).
# ROW_NUMBER() assigns 1..n within each duplicate group, so keeping rn = 1
# retains exactly one copy of each duplicate set.
window = Window.partitionBy("employee_id","employee_name","department","salary","manager_id").orderBy(col("employee_id").desc())

# df=df.withColumn("rn",row_number().over(window))  # tag each row with its rank inside its duplicate group
# df=df.filter(col("rn")==1)  # keep only the first row per group — drops the exact duplicates

# The 'rn' helper column is now redundant but harmless; drop it if you want a clean result:
# df = df.drop("rn")

df = df.withColumn("rn",row_number().over(window)).filter(col("rn")==1).drop("rn")

df.show()

+-----------+-------------+----------+------+----------+
|employee_id|employee_name|department|salary|manager_id|
+-----------+-------------+----------+------+----------+
|        101|   John Smith|        IT|135000|       201|
|        102|Alice Johnson|        IT| 85000|       201|
|        103| Bob Williams|        HR| 70000|       202|
|        104|  Carol Brown|        HR| 75000|       202|
|        105| David Miller|   Finance| 90000|       203|
|        106|   Emma Davis|   Finance| 80000|       203|
|        107| Frank Wilson|        IT| 85000|       201|
|        108| Grace Taylor|     Sales|120000|       204|
|        109|  Henry Moore|     Sales| 68000|       204|
|        110| Ivy Anderson| Marketing| 76000|       205|
|        201|Michael Scott|        IT|130000|       301|
|        202| Sarah Connor|        HR|120000|       301|
|        203|  Robert King|   Finance|145000|       301|
|        204|  Nancy White|     Sales|118000|       301|
|        205|  James Clark| Mar

##### Find Second Highest Salary

In [0]:
# Find the Second Highest Salary in Each Department (PySpark equivalent of the SQL approach above)
from pyspark.sql.functions import *  # gives col(), dense_rank(), etc.
from pyspark.sql.window import Window

# DENSE_RANK() (not ROW_NUMBER or RANK) so tied salaries share the same rank
# and no ranks are skipped — e.g. two people on the top salary both get rank 1,
# and the next distinct salary gets rank 2.
# Partition by department so the ranking restarts for each department.
window = Window.partitionBy("department").orderBy(col("salary").desc())

# Tag each row with its dense rank, keep only rank 2 (the second highest salary),
# and drop the helper column 'rn' from the final result.
df_new = df.withColumn("rn",dense_rank().over(window)).filter(col("rn")==2).drop("rn")
df=df.drop("rn")

df_new.show()

+-----------+-------------+----------+------+----------+
|employee_id|employee_name|department|salary|manager_id|
+-----------+-------------+----------+------+----------+
|        105| David Miller|   Finance| 90000|       203|
|        104|  Carol Brown|        HR| 75000|       202|
|        201|Michael Scott|        IT|130000|       301|
|        110| Ivy Anderson| Marketing| 76000|       205|
|        204|  Nancy White|     Sales|118000|       301|
+-----------+-------------+----------+------+----------+



##### Running Total

In [0]:
# Running Total of Salaries per Department (cumulative sum ordered by salary)
from pyspark.sql.functions import sum
from pyspark.sql.window import Window

# Partition by department so the running total restarts for each department.
# Order by salary (ascending) so salaries are accumulated lowest-to-highest.
# rowsBetween(unboundedPreceding, currentRow) defines a cumulative window frame:
# every row sees all rows from the start of its partition up to and including itself.
window = (
    Window.partitionBy("department")
    .orderBy("salary")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# sum("salary") over the cumulative window = running total:
# each row's running_total equals the sum of salaries from the partition's start through the current row.
df = df.withColumn("running_total", sum("salary").over(window))

df.show()

+-----------+-------------+----------+------+----------+-------------+
|employee_id|employee_name|department|salary|manager_id|running_total|
+-----------+-------------+----------+------+----------+-------------+
|        301|    Dilip M G|       CEO|140000|      NULL|       140000|
|        106|   Emma Davis|   Finance| 80000|       203|        80000|
|        302|Emma Davis Jr|   Finance| 80000|       203|       160000|
|        105| David Miller|   Finance| 90000|       203|       250000|
|        203|  Robert King|   Finance|145000|       301|       395000|
|        103| Bob Williams|        HR| 70000|       202|        70000|
|        104|  Carol Brown|        HR| 75000|       202|       145000|
|        202| Sarah Connor|        HR|120000|       301|       265000|
|        102|Alice Johnson|        IT| 85000|       201|        85000|
|        107| Frank Wilson|        IT| 85000|       201|       170000|
|        201|Michael Scott|        IT|130000|       301|       300000|
|     